# 02. YOLO 직접 추론 — 라이브러리 없이 전처리 · 출력 해석 · NMS

edge-lab 서버는 `ultralytics` 가 이 과정을 대신한다. 여기서는 **OpenVINO 로 모델만 돌리고 나머지는 전부 직접** 짠다.

1. letterbox — 비율을 지키며 640 x 640 에 맞추기
2. 출력 `[1, 84, 8400]` 읽기 — 후보 8400개, 후보마다 상자 4개 + 80종 점수
3. 점수 거르기 → NMS(겹친 상자 지우기) → 원래 사진 좌표로 되돌리기
4. 단계별 시간 재기

## 1. 준비

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

def pick(*prefer):
    """먼저 적은 장치부터, 이 PC 에 있는 것을 고른다."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

In [ ]:
YOLO_DIR = MODELS / "object" / "yolo11m_openvino_model"
xml = next(YOLO_DIR.glob("*.xml"))                 # ultralytics 가 내보낸 IR (yolo11m.xml)
model = core.read_model(xml)
print("입력:", model.input(0).partial_shape, "| 출력:", model.output(0).partial_shape)

# 클래스 이름 — 내보낼 때 같이 저장된 metadata.yaml 에 있다
import yaml
meta = yaml.safe_load(open(YOLO_DIR / "metadata.yaml", encoding="utf-8"))
NAMES = meta["names"]                              # {0: 'person', 1: 'bicycle', ...}
print(len(NAMES), "종:", list(NAMES.values())[:8], "...")

## 2. 사진

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. letterbox — 비율을 지키며 640 x 640 에 넣기
그냥 늘리면 사람이 뚱뚱해지거나 홀쭉해진다. 긴 쪽을 640 에 맞추고, 남는 곳은 회색(114)으로 채운다.
나중에 좌표를 되돌리려면 **배율**과 **여백**을 기억해 둬야 한다.

In [ ]:
SIZE = 640

def letterbox(bgr, size=SIZE):
    h, w = bgr.shape[:2]
    r = min(size / h, size / w)                     # 배율
    nw, nh = round(w * r), round(h * r)
    img = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
    px, py = (size - nw) / 2, (size - nh) / 2        # 좌우 · 위아래 여백 (가운데 정렬)
    top, bottom = round(py - 0.1), round(py + 0.1)
    left, right = round(px - 0.1), round(px + 0.1)
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))
    return img, r, (left, top)

boxed, ratio, pad = letterbox(frame)
print("배율:", round(ratio, 3), "여백(왼쪽, 위):", pad, "→", boxed.shape)
show(boxed, 400)

In [ ]:
def to_blob(img):
    x = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)          # YOLO 는 RGB 로 배웠다
    x = x.astype(np.float32) / 255.0                  # 0~255 → 0~1
    x = x.transpose(2, 0, 1)[None]                    # HWC → NCHW
    return np.ascontiguousarray(x)

blob = to_blob(boxed)
print(blob.shape, blob.dtype, "값 범위:", blob.min(), "~", blob.max())

## 4. 추론 — 출력 읽기

In [ ]:
DEVICE = pick("GPU", "NPU", "CPU")
compiled = core.compile_model(model, DEVICE)
out = compiled(blob)[compiled.output(0)]              # [1, 84, 8400]
print(DEVICE, "출력:", out.shape)

pred = out[0].T                                       # [8400, 84] — 줄 하나가 후보 하나
boxes_cxcywh = pred[:, :4]                            # 가운데 x, 가운데 y, 너비, 높이 (640 좌표)
scores_all = pred[:, 4:]                              # 80종 점수
cls = scores_all.argmax(1)                            # 가장 높은 종류
conf = scores_all.max(1)                              # 그 점수
print("후보", len(pred), "개 · 점수 0.25 넘는 후보:", int((conf > 0.25).sum()), "개")

## 5. 거르기 → NMS
같은 물체에 상자가 여러 개 겹쳐 나온다. **NMS**: 점수 높은 상자부터 고르고, 그 상자와 많이 겹치는(IoU 가 큰) 상자를 지운다.

IoU = 두 상자가 겹친 넓이 ÷ 두 상자를 합친 넓이 (0 = 안 겹침, 1 = 똑같음)

In [ ]:
def iou(a, b):
    """a: [4] 상자 하나, b: [N, 4] 상자들 (x1, y1, x2, y2)"""
    x1 = np.maximum(a[0], b[:, 0]); y1 = np.maximum(a[1], b[:, 1])
    x2 = np.minimum(a[2], b[:, 2]); y2 = np.minimum(a[3], b[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda r: (r[..., 2] - r[..., 0]) * (r[..., 3] - r[..., 1])
    return inter / (area(a) + area(b) - inter + 1e-9)

def nms(xyxy, scores, iou_th=0.45):
    order = scores.argsort()[::-1]                    # 점수 높은 순
    keep = []
    while len(order):
        i = order[0]
        keep.append(i)
        rest = order[1:]
        order = rest[iou(xyxy[i], xyxy[rest]) < iou_th]  # 많이 겹친 것은 버린다
    return np.array(keep, dtype=int)

def decode(pred, conf_th=0.25, iou_th=0.45):
    scores_all = pred[:, 4:]
    cls, conf = scores_all.argmax(1), scores_all.max(1)
    m = conf > conf_th                                # ① 점수 거르기
    b, cls, conf = pred[m, :4], cls[m], conf[m]
    xyxy = np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,   # 가운데·크기 → 모서리 좌표
                     b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], 1)
    # ② 종류별로 따로 NMS — 사람 상자가 자전거 상자를 지우면 안 된다.
    #    종류마다 좌표를 멀리 밀어 두면 한 번의 NMS 로 종류별 NMS 가 된다
    keep = nms(xyxy + cls[:, None] * 4096.0, conf, iou_th)
    return xyxy[keep], cls[keep], conf[keep], int(m.sum())

xyxy, cls_k, conf_k, n_pass = decode(pred)
print(f"8400 → 점수 통과 {n_pass} → NMS 뒤 {len(xyxy)}")

## 6. 원래 사진 좌표로 되돌리고 그리기

In [ ]:
def unletterbox(xyxy, ratio, pad, shape):
    h, w = shape[:2]
    out = xyxy.copy()
    out[:, [0, 2]] = (out[:, [0, 2]] - pad[0]) / ratio
    out[:, [1, 3]] = (out[:, [1, 3]] - pad[1]) / ratio
    out[:, [0, 2]] = out[:, [0, 2]].clip(0, w)
    out[:, [1, 3]] = out[:, [1, 3]].clip(0, h)
    return out

boxes = unletterbox(xyxy, ratio, pad, frame.shape)
view = frame.copy()
for (x1, y1, x2, y2), c, s in zip(boxes.astype(int), cls_k, conf_k):
    color = tuple(int(v) for v in np.random.default_rng(int(c)).integers(60, 255, 3))
    cv2.rectangle(view, (x1, y1), (x2, y2), color, 2)
    cv2.putText(view, f"{NAMES[int(c)]} {s:.2f}", (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    print(f"{NAMES[int(c)]:14s} {s:.2f}  ({x1}, {y1}) - ({x2}, {y2})")
show(view)

## 7. 어디에 시간이 드나

In [ ]:
def timed(n=20):
    t = {"전처리": 0.0, "추론": 0.0, "후처리": 0.0}
    for _ in range(n):
        t0 = time.perf_counter(); b, r, p = letterbox(frame); x = to_blob(b)
        t1 = time.perf_counter(); o = compiled(x)[compiled.output(0)]
        t2 = time.perf_counter(); decode(o[0].T)
        t3 = time.perf_counter()
        t["전처리"] += t1 - t0; t["추론"] += t2 - t1; t["후처리"] += t3 - t2
    return {k: v / n * 1000 for k, v in t.items()}

for k, v in timed().items():
    print(f"{k:6s} {v:7.2f} ms  {'█' * max(1, round(v))}")

## 해 볼 것
1. `decode(pred, conf_th=0.5)` · `iou_th=0.7` 로 바꿔 보자. 상자 수가 어떻게 달라지나?
2. letterbox 대신 `cv2.resize(frame, (640, 640))` 로 늘리면 결과가 어떻게 달라지나?
3. `DEVICE` 를 `"CPU"` · `"NPU"` 로 바꿔 7번 표를 비교해 보자. 전처리 · 후처리는 장치와 상관없이 CPU 가 한다.
4. 맞는지 확인: `from ultralytics import YOLO; r = YOLO(str(YOLO_DIR), task="detect")(frame)[0]; r.boxes` 와 비교해 보자.
5. 포즈 모델(`yolo11m-pose_openvino_model`)의 출력은 `[1, 56, 8400]` 이다 — 상자 4 + 점수 1 + 관절 17개 × (x, y, 신뢰도).
   관절까지 그려 보자.